This session demonstrates:

1) What LangGraph is
2) How agent workflows are graphs
3) Nodes, edges, and state
4) A working LangGraph agent
5) A small finance workflow example


# Session 4 – Building Agents with LangGraph

In this notebook we will build our first **LangGraph agent workflow**.

LangGraph allows us to design **stateful AI agent systems** using graphs.

Instead of linear pipelines, agents can:

- reason
- call tools
- loop
- update state
- decide next steps

Projects in this notebook:

1. Understanding LangGraph
2. Building a simple graph workflow
3. Creating an LLM node
4. Adding a tool node
5. Building a finance research agent

In [1]:
!pip install langchain==0.2.14 langchain-core==0.2.32 langchain-groq==0.1.9

['Collecting langchain==0.2.14',
 '  Downloading langchain-0.2.14-py3-none-any.whl.metadata (7.1 kB)',
 'Collecting langchain-core==0.2.32',
 '  Downloading langchain_core-0.2.32-py3-none-any.whl.metadata (6.2 kB)',
 'Collecting langchain-groq==0.1.9',
 '  Downloading langchain_groq-0.1.9-py3-none-any.whl.metadata (2.9 kB)',
 'Requirement already satisfied: PyYAML>=5.3 in /usr/local/lib/python3.12/dist-packages (from langchain==0.2.14) (6.0.3)',
 'Requirement already satisfied: SQLAlchemy<3,>=1.4 in /usr/local/lib/python3.12/dist-packages (from langchain==0.2.14) (2.0.48)',
 'Requirement already satisfied: aiohttp<4.0.0,>=3.8.3 in /usr/local/lib/python3.12/dist-packages (from langchain==0.2.14) (3.13.3)',
 'Collecting langchain-text-splitters<0.3.0,>=0.2.0 (from langchain==0.2.14)',
 '  Downloading langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)',
 'Collecting langsmith<0.2.0,>=0.1.17 (from langchain==0.2.14)',
 '  Downloading langsmith-0.1.147-py3-none-any.whl.metada

In [1]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [2]:
os.environ["GROQ_API_KEY"] = "<your-api-key>"

In [3]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
#This is the reasoning engine inside the agent.

Understanding Langgraph State
## What is State?

State is a shared data structure that moves through the agent workflow.

Each node can:

- read state
- update state
- pass it forward

In [4]:
# defining state schema
class AgentState(TypedDict):
    question: str
    answer: str

## LLM Node

This node will use the language model to answer a question.

In [5]:
def llm_node(state):

    question = state["question"]

    response = llm.invoke(question)

    return {"answer": response.content}

In [6]:
# Build graph
builder = StateGraph(AgentState)

In [7]:
# add node
builder.add_node("llm_node", llm_node)

In [8]:
# set entry point
builder.set_entry_point("llm_node")

In [9]:
# set end point
builder.add_edge("llm_node", END)

In [10]:
# compile graph
graph = builder.compile()

In [11]:
# run graph
graph.invoke({"question":"Why are tech stocks falling today?"})

{'question': 'Why are tech stocks falling today?',
 'answer': '**Quick Take:**  \nTech stocks are under pressure today across the major U.S. indexes (NASDAQ‑100, S&P\u202f500 Information‑Technology sector, and the Russell\u202f1000\u202fTech). The slide is being driven by a mix of macro‑economic data, earnings‑season dynamics, and a few sector‑specific headlines that have spooked investors.\n\nBelow is a concise rundown of the most salient factors that are likely behind today’s pull‑back, followed by a “what to watch next” checklist and some practical tips for anyone with exposure to tech equities.\n\n---\n\n## 1. Macro‑Economic Headwinds\n\n| Factor | What happened | Why it matters for tech |\n|--------|---------------|------------------------|\n| **Federal Reserve policy signal** | In a post‑meeting press conference, Fed Chair **Lena Ortiz** hinted that the central bank may **raise rates again** later this year if inflation doesn’t cool faster than expected. The “higher‑for‑longer” n

In [12]:
# Add Tool Node
def stock_price_tool(symbol):

    return f"The simulated price of {symbol} is $180"

In [13]:
# Update State
class AgentState(TypedDict):
    question: str
    stock_price: str

In [15]:
# Tool Node
def tool_node(state):

    result = stock_price_tool("NVDA")

    return {"stock_price": result}

In [16]:
# Build Tool Graph
builder = StateGraph(AgentState)

builder.add_node("tool_node", tool_node)

builder.set_entry_point("tool_node")

builder.add_edge("tool_node", END)

graph = builder.compile()

In [17]:
# Run Tool Graph
graph.invoke({"question":"What is Nvidia's stock price?"})

{'question': "What is Nvidia's stock price?",
 'stock_price': 'The simulated price of NVDA is $180'}

Finance Research Agent
Now we build a small quant research agent workflow.
Steps:
User asks question
LLM analyzes market
Agent suggests hypotheses

In [18]:
# update State
class ResearchState(TypedDict):
    question: str
    hypothesis: str

In [19]:
# hypothesis node
def hypothesis_node(state):

    prompt = f"""
Suggest three possible reasons why the following stock might be dropping.

Question: {state["question"]}
"""

    response = llm.invoke(prompt)

    return {"hypothesis": response.content}

In [20]:
# Build Research graph
builder = StateGraph(ResearchState)

builder.add_node("hypothesis_node", hypothesis_node)

builder.set_entry_point("hypothesis_node")

builder.add_edge("hypothesis_node", END)

research_graph = builder.compile()

In [21]:
# Run Research Agent
research_graph.invoke(
{
"question":"Why might Nvidia stock be falling today?"
}
)
# This simulates a quant research assistant.

{'question': 'Why might Nvidia stock be falling today?',
 'hypothesis': '**Three common reasons a stock like Nvidia could be slipping on a given day**\n\n| # | Possible driver | How it could push the price lower |\n|---|----------------|-----------------------------------|\n| 1 | **Disappointing earnings or guidance** | If Nvidia’s quarterly results miss analysts’ expectations—whether it’s lower‑than‑expected revenue, weaker margins, or a cut to future sales guidance—investors may rush to sell. A surprise dip in key metrics (e.g., GPU shipments, data‑center sales, or AI‑related services) can trigger profit‑taking and a short‑term price decline, even if the longer‑term outlook remains strong. |\n| 2 | **Broader macro‑economic headwinds** | A rise in interest rates, a slowdown in tech spending, or heightened recession fears can hurt high‑growth, high‑valuation names like Nvidia. Higher borrowing costs make future cash‑flow projections look less attractive, and investors may rotate out of

Visualsing Agent Workflow
User Question
     ↓
LLM Node
     ↓
Tool Node
     ↓
Research Node
     ↓
Final Output

Why Langgraph Matters?
LangGraph enables production AI agents because it supports:

Stateful reasoning

Multi-step workflows

Tool execution

Multi-agent collaboration


Challenge:

Modify the research agent to also analyze news sentiment before generating hypotheses.